In [52]:
# Search_All_Cemeteries.py
# 5/10/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(5, 10)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************
my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Search_All_Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"



fileIn="AllCemeteries_CountyCorrelated"
fileOutput="AllCemeteries_CountyCorrelated"

if append==1:fileIn='AllCemeteries_CountyCorrelated_X'


fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\Graves\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Graves\\"+fileOutput+"_Interim.csv"

cemetery_info_df=pd.read_csv(fileCemeteriesIn, encoding='latin1')
cemetery_id_list=cemetery_info_df["Cemetery_ID"]
cemetery_name_list=cemetery_info_df["Cemetery_Name"]
cemetery_city_list=cemetery_info_df["city"]
cemetery_county_list=cemetery_info_df["county"]
cemetery_state_list=cemetery_info_df["state"]
cemetery_county_id_list=cemetery_info_df["county_id"]
cemetery_state_id_list=cemetery_info_df["state_id"]


field_list=['lastName','firstName','birthYear','deathYear','cemeteryName','cemeteryId','cityName','stateName']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=1*60
date_df=pd.DataFrame([])
get_counter=0
cemetery_count=0
total_cemetery_count=len(cemetery_id_list)
start_year=1890
last_year=1950
for cemetery_id in cemetery_id_list:
    url0 = "https://www.findagrave.com/cemetery/"
    url1 ="/memorial-search?ajax=true&skip="
    url2 ="&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear="
    url2a ="&deathyearfilter=exact&locationId=&location=&memorialid=&datefilter=&orderby=b&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=true&linkedToName=&fuzzyNames=&mcid="

    cemetery_name=cemetery_name_list.iloc[cemetery_count]
    cemetery_city=cemetery_city_list.iloc[cemetery_count]
    cemetery_county=cemetery_county_list.iloc[cemetery_count]
    cemetery_state=cemetery_state_list.iloc[cemetery_count]
    cemetery_county_id=cemetery_county_id_list.iloc[cemetery_count]
    cemetery_state_id=cemetery_state_id_list.iloc[cemetery_count]

    cemetery_count+=1
    interval=10
    search_year=start_year
    cemetery_grave_count=0
    break_flag=0
    while  search_year<=last_year:
        frame_count=0
        grave_frame_count=0
        grave_count=100
        while grave_frame_count<grave_count :
            frame_count+=1
            url=url0+str(cemetery_id)+url1+str(grave_frame_count)+url2+str(search_year)+url2a
            capture_json,grave_count=get_grave_count(url)
            if status_code!=200:print('\tstatus_code:',status_code, 'frame count:',frame_count);break_flag=1;break 
            # print('Point 1:search_year,grave_frame_count,grave_count:',search_year,grave_frame_count,grave_count)
            
            if grave_count>10000 and interval==1:continue
            print(url)
            print(json.dumps(capture_json))            
            input('capture_json')
            
            grave_frame_df=extract_grave_data(capture_json)

            if len(grave_frame_df)==0:search_year+=2*interval;break
            graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
            grave_frame_count+=len(grave_frame_df)
            cemetery_grave_count+=len(grave_frame_df)
            graves_df=clean_dataframe(graves_df)
            graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
            print(f"    --- {process_name:<20}• Cemetery: {cemetery_name}•{cemetery_city}•{cemetery_state} • Search Year: {search_year} • grave count: {grave_frame_count}/{grave_count} ")

            get_counter+=1
    
            sleep_time=random_sleep(get_counter)       
            time.sleep(sleep_time)
        if break_flag==1: break
        search_year+=2*interval
    print(f"***{process_name:<20}• Cemetery: {cemetery_name}•{cemetery_city}•{cemetery_state} • Cemetery Grave Count: {cemetery_grave_count} • Cemetery count: {cemetery_count}/{total_cemetery_count} \n")
    # print(graves_df[['lastName',  'firstName','cohen_count']])
    # input('')
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Search_All_Cemeteries   *** , Time: 2025-05-11 06:32:36.658678  **

	status_code: 404 url https://www.findagrave.com/cemetery/Green Family Cemetery/memorial-search?ajax=true&skip=0&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear=1890&deathyearfilter=exact&locationId=&location=&memorialid=&datefilter=&orderby=b&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=true&linkedToName=&fuzzyNames=&mcid=
	status_code: 404 frame count: 1
***Search_All_Cemeteries • Cemetery: 2412308•Montgomery•Alabama • Cemetery Grave Count: 0 • Cemetery count: 1/96667 

	status_code: 404 url https://www.findagrave.com/cemetery/Waller Cemetery/memorial-search?ajax=true&skip=0&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear=1890&deathyearfilter=exact&locationId=&location=&memorialid=&datefilte

KeyboardInterrupt: 

In [43]:
# Find_Problem_line_Input_File.py

def find_problematic_line(filename, problem_position):
    with open(filename, 'rb') as file:
        # Start from the beginning of the file
        file.seek(0)
        
        line_number = 1
        current_position = 0
        problematic_line = None
        problematic_line_bytes = None
        
        # Read the file line by line
        for line in file:
            line_length = len(line)
            
            # Check if our problem position falls within this line
            if current_position <= problem_position < current_position + line_length:
                problematic_line_bytes = line
                # Try different encodings to display the line
                encodings = ['utf-8', 'latin1', 'cp1252', 'ISO-8859-1']
                
                for encoding in encodings:
                    try:
                        problematic_line = line.decode(encoding, errors='replace')
                        break
                    except:
                        continue
                
                # Calculate the position within the line
                position_in_line = problem_position - current_position
                
                return {
                    'line_number': line_number,
                    'line_content': problematic_line,
                    'line_bytes': problematic_line_bytes,
                    'position_in_line': position_in_line,
                    'problem_byte': line[position_in_line:position_in_line+1].hex()
                }
            
            current_position += line_length
            line_number += 1
        
        return None

# Use the function
result = find_problematic_line(fileCemeteriesIn, 79751)

if result:
    print(f"Problem found on line {result['line_number']}")
    print(f"Problematic byte (hex): 0x{result['problem_byte']}")
    print(f"Position in line: {result['position_in_line']}")
    print("\nLine content:")
    print(result['line_content'])
    
    # Show where the problem is with a pointer
    if result['line_content']:
        pointer_line = ' ' * result['position_in_line'] + '^'
        print(pointer_line)
else:
    print("Could not locate the problematic line")

Problem found on line 936
Problematic byte (hex): 0x6f
Position in line: 37

Line content:
2270736,Ridgecrest Pet Cemetery,Jackson,Madison County,Tennessee,county_2481,state_45

                                     ^
